# 🚀 SpaceX Falcon 9 Landing Prediction
## Notebook 5 — EDA with SQL

We load the SpaceX dataset into a local SQLite database and answer
ten business questions purely with SQL.


In [5]:
%pip install ipython-sql sqlalchemy pandas -q
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'


Note: you may need to restart the kernel to use updated packages.
The sql extension is already loaded. To reload it, use:
  %reload_ext sql



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import sqlite3
import pandas as pd

con = sqlite3.connect("spacex.db")
%sql sqlite:///spacex.db


In [7]:
# IBM-hosted Spacex.csv — the course's canonical SQL dataset
spacex_sql_df = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud"
    "/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
)
spacex_sql_df.to_sql("LAUNCHES", con, if_exists='replace', index=False, method='multi')
print(f"Rows loaded: {len(spacex_sql_df)}")
spacex_sql_df.head(3)


Rows loaded: 101


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt


### Q1 — First 10 rows

In [8]:
%%sql
SELECT * FROM LAUNCHES LIMIT 10;


 * sqlite:///spacex.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt
2013-09-29,16:00:00,F9 v1.1 B1003,VAFB SLC-4E,CASSIOPE,500,Polar LEO,MDA,Success,Uncontrolled (ocean)
2013-12-03,22:41:00,F9 v1.1,CCAFS LC-40,SES-8,3170,GTO,SES,Success,No attempt
2014-01-06,22:06:00,F9 v1.1,CCAFS LC-40,Thaicom 6,3325,GTO,Thaicom,Success,No attempt
2014-04-18,19:25:00,F9 v1.1,CCAFS LC-40,SpaceX CRS-3,2296,LEO (ISS),NASA (CRS),Success,Controlled (ocean)
2014-07-14,15:15:00,F9 v1.1,CCAFS LC-40,OG2 Mission 1 6 Orbcomm-OG2 satellites,1316,LEO,Orbcomm,Success,Controlled (ocean)


### Q2 — All distinct launch sites

In [9]:
%%sql
SELECT DISTINCT "Launch Site" FROM LAUNCHES;


 * sqlite:///spacex.db
Done.


"""Launch Site"""
Launch Site


### Q3 — First 5 launches from KSC LC-39A

In [10]:
%%sql
SELECT *
FROM LAUNCHES
WHERE "Launch Site" LIKE 'KSC LC-39A%'
LIMIT 5;


 * sqlite:///spacex.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome


### Q4 — Total payload mass carried per NASA customer

In [11]:
%%sql
SELECT Customer,
       SUM("PAYLOAD_MASS__KG_") AS TotalPayload_kg
FROM LAUNCHES
WHERE Customer LIKE '%NASA%'
GROUP BY Customer
ORDER BY TotalPayload_kg DESC;


 * sqlite:///spacex.db
Done.


Customer,TotalPayload_kg
NASA (CRS),45596
NASA (CCDev),12530
NASA (CCP),12500
NASA (CCD),12055
NASA (CTS),12050
Iridium Communications GFZ ‚ NASA,6460
"NASA (CRS), Kacific 1",2617
NASA / NOAA / ESA / EUMETSAT,1192
U.S. Air Force NASA NOAA,570
NASA (LSP) NOAA CNES,553


### Q5 — Average payload mass grouped by booster version

In [12]:
%%sql
SELECT "Booster Version",
       ROUND(AVG("PAYLOAD_MASS__KG_"), 1) AS AvgPayload_kg,
       COUNT(*) AS Launches
FROM LAUNCHES
GROUP BY "Booster Version"
ORDER BY AvgPayload_kg DESC;


 * sqlite:///spacex.db
Done.


"""Booster Version""",AvgPayload_kg,Launches
Booster Version,6138.3,101


### Q6 — First successful drone-ship landing

In [13]:
%%sql
SELECT "Date", "Booster Version", "Launch Site", "Landing Outcome"
FROM LAUNCHES
WHERE "Landing Outcome" = 'Success (drone ship)'
ORDER BY "Date" ASC
LIMIT 1;


 * sqlite:///spacex.db
Done.


Date,"""Booster Version""","""Launch Site""","""Landing Outcome"""


### Q7 — All boosters with successful ground-pad landings

In [14]:
%%sql
SELECT DISTINCT "Booster Version"
FROM LAUNCHES
WHERE "Landing Outcome" = 'Success (ground pad)';


 * sqlite:///spacex.db
Done.


"""Booster Version"""


### Q8 — Landing outcome breakdown between two dates

In [15]:
%%sql
SELECT "Landing Outcome", COUNT(*) AS Count
FROM LAUNCHES
WHERE "Date" BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY "Landing Outcome"
ORDER BY Count DESC;


 * sqlite:///spacex.db
Done.


"""Landing Outcome""",Count
Landing Outcome,31


### Q9 — Top 10 heaviest payloads on successful missions

In [16]:
%%sql
SELECT "Booster Version", "Launch Site", "PAYLOAD_MASS__KG_"
FROM LAUNCHES
WHERE "Mission Outcome" = 'Success'
ORDER BY "PAYLOAD_MASS__KG_" DESC
LIMIT 10;


 * sqlite:///spacex.db
Done.


"""Booster Version""","""Launch Site""",PAYLOAD_MASS__KG_


### Q10 — Success rate per launch site

In [17]:
%%sql
SELECT
    "Launch Site",
    COUNT(*) AS TotalLaunches,
    SUM(CASE WHEN "Mission Outcome" = 'Success' THEN 1 ELSE 0 END) AS Successes,
    ROUND(
        100.0 * SUM(CASE WHEN "Mission Outcome" = 'Success' THEN 1 ELSE 0 END) / COUNT(*),
        1
    ) AS SuccessRatePct
FROM LAUNCHES
GROUP BY "Launch Site"
ORDER BY SuccessRatePct DESC;


 * sqlite:///spacex.db
Done.


"""Launch Site""",TotalLaunches,Successes,SuccessRatePct
Launch Site,101,0,0.0
